In [ ]:
import os
from glob import glob
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
rho = 917

In [ ]:
def get_val(fname):
    val = fname.split('_')[-1]
    return int(val)*1e-4

In [ ]:
def process_run(forc):
    folder = f'../results/calib/{forc}*'

    #Get all folders
    fnames = sorted(glob(folder))

    #Get p1 values
    p1 = [get_val(fname) for fname in fnames]
    p2 = [0]

    print('Starting',forc,p1,'...')

    # Set up main domain
    ds0 = xr.open_dataset('/home/erwin/data/ismip7/parameterisations/ocean/bfrns/BFRN_ismip4km_v2.nc')
    x = ds0.x
    y = ds0.y

    dims = ['p1','p2','y','x']
    coords = {
        'p1': p1,
        'p2': p2,
        'y': y,
        'x': x
    }

    dsm = xr.Dataset({
        'melt_rate': xr.DataArray(
            data = np.zeros((len(p1),len(p2),len(y),len(x))),
            dims=dims,
            coords={'p1':p1,'p2':p2,'y':y,'x':x},
            attrs={'units':'kg/m2/y'}
        )
    })

    for f,fname in enumerate(fnames):
        ds = xr.open_dataset(os.path.join(fname,'laddie_output_grid.nc'))
        ds = ds.isel(time=-1).drop_vars('time')
        if forc[:8] == 'Dutrieux': 
            ds = ds.reindex_like(ds0, method=None, fill_value=0)
        dsm['melt_rate'][f,:,:,:] = ds['melt'].copy() * rho *3600*24*365.25
        ds.close()

    dsm.to_netcdf(f'../results/combined/{forc}.nc')

    ds0.close()
    dsm.close()

    print('Finished',forc,p1)

In [ ]:
process_run('climatology')
process_run('Mathiot_NEMO_cold_v2')
process_run('Mathiot_NEMO_warm_v2')
process_run('Naughten_FESOM_ACCESS_cold')
process_run('Naughten_FESOM_ACCESS_warm')
for y in [1994,2000,2006,2007,2009,2010,2011,2012,2014,2016,2018,2019,2020]:
    process_run(f'Dutrieux_{y}')

In [ ]:
ds = xr.open_dataset('../results/calib/Dutrieux_2009_08/laddie_output_grid.nc').isel(time=-1)

fig,ax = plt.subplots(1,2,sharex=True,sharey=True,figsize=(7,4))
ax[0].pcolormesh(ds.x,ds.y,ds.melt.where(ds.melt>0,other=np.nan))
ax[0].axvline(-1.625e6,0,1,c='tab:orange')
mmelt = ds.melt.where(ds.x>-1.625e6,other=0)
im = ax[1].pcolormesh(ds.x,ds.y,mmelt.where(mmelt>0,other=np.nan))
plt.colorbar(im,ax=ax[1])